In [2]:
from pathlib import Path
import pandas as pd

In [3]:
data_dir = Path("../data/02_interim/freddie_mac/2015")

In [4]:
orig = pd.read_parquet(data_dir / "origination.parquet")
perf = pd.read_parquet(data_dir / "performance")

In [6]:
print(orig.shape)
perf.shape

(50000, 31)


(3467166, 35)

In [7]:
perf["current_loan_delinquency_status"].value_counts(dropna=False).sort_index()

current_loan_delinquency_status
00    3415433
01      23522
02       6686
03       3756
04       2597
       ...   
67          1
68          1
69          1
70          1
RA        567
Name: count, Length: 72, dtype: int64[pyarrow]

In [8]:
perf["zero_balance_code"].value_counts(dropna=False).sort_index()

zero_balance_code
01        38212
02           45
03           18
09           55
15           18
16          108
96           35
<NA>    3428675
Name: count, dtype: int64[pyarrow]

#

In [9]:
perf["delq_numeric"] = pd.to_numeric(
    perf["current_loan_delinquency_status"],
    errors="coerce",
)

In [ ]:
loan_max_delq = (
    perf.groupby("loan_id")["delq_numeric"].max().sort_values(ascending=False)
)

loan_max_delq.head(20)

loan_id
F15Q20286510    70
F15Q40056601    66
F15Q40098526    53
F15Q40096929    52
F15Q20023143    52
F15Q30144200    49
F15Q10122978    48
F15Q20346220    42
F15Q30342177    42
F15Q10336553    41
F15Q40166409    41
F15Q10039711    41
F15Q30337175    40
F15Q40266625    40
F15Q20134499    39
F15Q40296466    39
F15Q10234750    36
F15Q10324227    36
F15Q20368617    35
F15Q20309941    35
Name: delq_numeric, dtype: Int64

### Exploring Bad Loan Examples

In [11]:
serious_loans = loan_max_delq[loan_max_delq >= 3]

print("Loans reaching 90+ DPD:", len(serious_loans))
print("Share:", len(serious_loans) / perf["loan_id"].nunique())

Loans reaching 90+ DPD: 2049
Share: 0.04098


In [12]:
bad_loan_id = serious_loans.index[0]

orig.loc[orig["loan_id"] == bad_loan_id].T

,21208
credit_score,622
first_payment_date,2015-08
first_time_homebuyer_flag,N
maturity_date,2045-07
msa,<NA>
mi_percentage,0
number_of_units,1
occupancy_status,P
original_cltv,97
original_dti,999


In [15]:
columns = [
    "loan_id",
    "period",
    "loan_age",
    "current_actual_upb",
    "current_loan_delinquency_status",
    "zero_balance_code",
    "modification_flag",
]

bad_history = perf.loc[
    perf["loan_id"] == bad_loan_id,
    columns,
].sort_values("period")

bad_history

,loan_id,period,loan_age,current_actual_upb,current_loan_delinquency_status,zero_balance_code,modification_flag
1529860,F15Q20286510,2015-07,0,209000.0,00,<NA>,<NA>
1529861,F15Q20286510,2015-08,1,209000.0,00,<NA>,<NA>
1529862,F15Q20286510,2015-09,2,209000.0,00,<NA>,<NA>
1529863,F15Q20286510,2015-10,3,208000.0,00,<NA>,<NA>
1529864,F15Q20286510,2015-11,4,208000.0,00,<NA>,<NA>
...,...,...,...,...,...,...,...
1529968,F15Q20286510,2024-07,81,209594.69,66,<NA>,P
1529969,F15Q20286510,2024-08,82,209594.69,67,<NA>,P
1529970,F15Q20286510,2024-09,83,209594.69,68,<NA>,P
1529971,F15Q20286510,2024-10,84,209594.69,69,<NA>,P


## Exploring Healthy Loan Examples

In [16]:
healthy_loans = loan_max_delq[loan_max_delq == 0]

healthy_loan_id = healthy_loans.index[0]

In [17]:
orig.loc[orig["loan_id"] == healthy_loan_id].T

,3941
credit_score,788
first_payment_date,2015-04
first_time_homebuyer_flag,N
maturity_date,2045-03
msa,31084
mi_percentage,0
number_of_units,1
occupancy_status,P
original_cltv,76
original_dti,47


In [18]:
healthy_history = perf.loc[
    perf["loan_id"] == healthy_loan_id,
    columns,
].sort_values("period")

healthy_history

,loan_id,period,loan_age,current_actual_upb,current_loan_delinquency_status,zero_balance_code,modification_flag
277798,F15Q10118703,2015-03,0,349000.0,00,<NA>,<NA>
277799,F15Q10118703,2015-04,1,349000.0,00,<NA>,<NA>
277800,F15Q10118703,2015-05,2,349000.0,00,<NA>,<NA>
277801,F15Q10118703,2015-06,3,348000.0,00,<NA>,<NA>
277802,F15Q10118703,2015-07,4,347000.0,00,<NA>,<NA>
...,...,...,...,...,...,...,...
277860,F15Q10118703,2020-05,62,307468.83,00,<NA>,<NA>
277861,F15Q10118703,2020-06,63,306808.77,00,<NA>,<NA>
277862,F15Q10118703,2020-07,64,306146.65,00,<NA>,<NA>
277863,F15Q10118703,2020-08,65,305482.46,00,<NA>,<NA>


### Early Exit Loan

In [19]:
terminated = perf.loc[perf["zero_balance_code"].notna()]

terminated[
    [
        "loan_id",
        "period",
        "loan_age",
        "zero_balance_code",
        "zero_balance_effective_date",
        "current_loan_delinquency_status",
    ]
].head(20)

,loan_id,period,loan_age,zero_balance_code,zero_balance_effective_date,current_loan_delinquency_status
67,F15Q10000025,2020-10,67,01,2020-10,00
84,F15Q10000031,2016-07,16,01,2016-07,00
269,F15Q10000087,2019-05,51,01,2019-05,00
714,F15Q10000230,2018-10,44,01,2018-10,00
918,F15Q10000275,2020-11,69,01,2020-11,00
986,F15Q10000276,2020-09,67,01,2020-09,00
1047,F15Q10000299,2020-02,60,01,2020-02,00
1080,F15Q10000353,2017-11,32,01,2017-11,00
1158,F15Q10000407,2021-07,77,01,2021-07,00
1233,F15Q10000424,2021-04,74,01,2021-04,00


In [20]:
exit_loan_id = terminated.iloc[0]["loan_id"]

(
    perf.loc[
        perf["loan_id"] == exit_loan_id,
        columns,
    ].sort_values("period")
)

,loan_id,period,loan_age,current_actual_upb,current_loan_delinquency_status,zero_balance_code,modification_flag
0,F15Q10000025,2015-03,0,417000.0,00,<NA>,<NA>
1,F15Q10000025,2015-04,1,415000.0,00,<NA>,<NA>
2,F15Q10000025,2015-05,2,413000.0,00,<NA>,<NA>
3,F15Q10000025,2015-06,3,409000.0,00,<NA>,<NA>
4,F15Q10000025,2015-07,4,409000.0,00,<NA>,<NA>
...,...,...,...,...,...,...,...
63,F15Q10000025,2020-06,63,287968.69,00,<NA>,<NA>
64,F15Q10000025,2020-07,64,285788.11,00,<NA>,<NA>
65,F15Q10000025,2020-08,65,283602.99,00,<NA>,<NA>
66,F15Q10000025,2020-09,66,281413.32,00,<NA>,<NA>


### Terminated Loans

In [21]:
terminated = perf.loc[
    perf["zero_balance_code"].notna(),
    [
        "loan_id",
        "period",
        "loan_age",
        "current_actual_upb",
        "current_loan_delinquency_status",
        "zero_balance_code",
        "zero_balance_effective_date",
        "actual_loss",
    ],
]

terminated.head(20)

,loan_id,period,loan_age,current_actual_upb,current_loan_delinquency_status,zero_balance_code,zero_balance_effective_date,actual_loss
67,F15Q10000025,2020-10,67,0.0,00,01,2020-10,<NA>
84,F15Q10000031,2016-07,16,0.0,00,01,2016-07,<NA>
269,F15Q10000087,2019-05,51,0.0,00,01,2019-05,<NA>
714,F15Q10000230,2018-10,44,0.0,00,01,2018-10,<NA>
918,F15Q10000275,2020-11,69,0.0,00,01,2020-11,<NA>
986,F15Q10000276,2020-09,67,0.0,00,01,2020-09,<NA>
1047,F15Q10000299,2020-02,60,0.0,00,01,2020-02,<NA>
1080,F15Q10000353,2017-11,32,0.0,00,01,2017-11,<NA>
1158,F15Q10000407,2021-07,77,0.0,00,01,2021-07,<NA>
1233,F15Q10000424,2021-04,74,0.0,00,01,2021-04,<NA>


In [22]:
perf["zero_balance_code"].value_counts(dropna=False).sort_index()

zero_balance_code
01        38212
02           45
03           18
09           55
15           18
16          108
96           35
<NA>    3428675
Name: count, dtype: int64[pyarrow]

### Outlier Checks

In [23]:
perf_24m = perf.loc[perf["loan_age"].between(0, 24)].copy()

In [24]:
perf_24m["delq_numeric"] = pd.to_numeric(
    perf_24m["current_loan_delinquency_status"],
    errors="coerce",
)

In [25]:
perf_24m["is_90dpd"] = perf_24m["delq_numeric"].ge(3) | perf_24m[
    "current_loan_delinquency_status"
].eq("RA")

In [26]:
loan_target = (
    perf_24m.groupby("loan_id")
    .agg(
        ever_90dpd_24m=("is_90dpd", "max"),
        max_delinquency=("delq_numeric", "max"),
        months_observed=("loan_age", "nunique"),
        max_loan_age=("loan_age", "max"),
    )
    .reset_index()
)

In [27]:
loan_target["ever_90dpd_24m"].value_counts()

ever_90dpd_24m
False    48313
True       346
Name: count, dtype: Int64

In [28]:
loan_target["ever_90dpd_24m"].value_counts(normalize=True)

ever_90dpd_24m
False    0.992889
True     0.007111
Name: proportion, dtype: Float64

In [29]:
orig_loan_ids = set(orig["loan_id"])
target_loan_ids = set(loan_target["loan_id"])

missing_ids = orig_loan_ids - target_loan_ids

print("Origination loans:", len(orig_loan_ids))
print("Target loans:", len(target_loan_ids))
print("Missing loans:", len(missing_ids))

Origination loans: 50000
Target loans: 48659
Missing loans: 1341


In [32]:
missing_perf = perf[perf["loan_id"].isin(missing_ids)]

print("Missing loans found in performance:")
print(missing_perf["loan_id"].nunique())

print("\nLoan age distribution:")
print(missing_perf["loan_age"].describe())

print("\nSample:")

missing_perf[
        [
            "loan_id",
            "period",
            "loan_age",
            "current_loan_delinquency_status",
            "zero_balance_code",
        ]
    ].head(20)

Missing loans found in performance:
1341

Loan age distribution:
count      53207.0
mean      73.81837
std      24.996073
min           25.0
25%           54.0
50%           67.0
75%           93.0
max          133.0
Name: loan_age, dtype: Float64

Sample:


,loan_id,period,loan_age,current_loan_delinquency_status,zero_balance_code
889241,F15Q10358712,2017-07,29,00,<NA>
889242,F15Q10358712,2017-08,30,00,<NA>
889243,F15Q10358712,2017-09,31,00,<NA>
889244,F15Q10358712,2017-10,32,00,<NA>
889245,F15Q10358712,2017-11,33,00,<NA>
889246,F15Q10358712,2017-12,34,00,<NA>
889247,F15Q10358712,2018-01,35,00,<NA>
889248,F15Q10358712,2018-02,36,00,<NA>
889249,F15Q10358712,2018-03,37,00,<NA>
889250,F15Q10358712,2018-04,38,00,<NA>


In [33]:
orig.loc[
    orig["loan_id"] == "F15Q10358712",
    [
        "loan_id",
        "first_payment_date",
        "original_loan_term",
        "credit_score",
        "original_upb",
    ],
].T

,12192
loan_id,F15Q10358712
first_payment_date,2015-03
original_loan_term,360
credit_score,706
original_upb,203000


In [35]:
missing_summary = missing_perf.groupby("loan_id").agg(
    first_period=("period", "min"),
    first_loan_age=("loan_age", "min"),
    last_loan_age=("loan_age", "max"),
    n_observations=("period", "size"),
)

missing_summary["first_loan_age"].value_counts().sort_index()

first_loan_age
25     14
26      8
27      6
28      5
29      5
30      5
31      5
32      4
33      5
34      8
35     20
36     36
37     37
38     74
39     86
40     93
41     72
42     77
43    103
44     97
45     53
46     54
47     54
48     84
49     86
50     75
51     88
52     68
53     13
54      2
57      1
58      2
60      1
Name: count, dtype: Int64

In [36]:
loan_observation = (
    perf_24m.groupby("loan_id")
    .agg(
        first_loan_age=("loan_age", "min"),
        last_loan_age=("loan_age", "max"),
        n_months_observed=("loan_age", "nunique"),
    )
    .reset_index()
)

loan_observation["last_loan_age"].value_counts().sort_index()

last_loan_age
0        20
1        47
2        83
3       115
4       153
5       220
6       260
7       350
8       372
9       341
10      384
11      386
12      424
13      464
14      472
15      414
16      437
17      408
18      403
19      405
20      339
21      379
22      349
23      366
24    41068
Name: count, dtype: Int64

In [38]:
early_end = loan_observation[loan_observation["last_loan_age"] < 24]

print("Loans ending before 24 months:", len(early_end))

Loans ending before 24 months: 7591


In [40]:
early_end_ids = set(early_end["loan_id"])

early_end_perf = perf[perf["loan_id"].isin(early_end_ids)].copy()

early_end_last = (
    early_end_perf.sort_values(["loan_id", "period"]).groupby("loan_id").tail(1)
)

early_end_last["zero_balance_code"].value_counts(dropna=False).sort_index()

zero_balance_code
01      7547
02         3
03         4
09         6
15         1
96        29
<NA>       1
Name: count, dtype: int64[pyarrow]

In [42]:
adverse_zbc_ids = set(
    early_end_last.loc[
        early_end_last["zero_balance_code"].isin(["02", "03", "09"]),
        "loan_id",
    ]
)

adverse_check = perf_24m.loc[
    perf_24m["loan_id"].isin(adverse_zbc_ids),
    [
        "loan_id",
        "loan_age",
        "current_loan_delinquency_status",
        "zero_balance_code",
    ],
].sort_values(["loan_id", "loan_age"])

adverse_check

,loan_id,loan_age,current_loan_delinquency_status,zero_balance_code
172676,F15Q10074428,0,00,<NA>
172677,F15Q10074428,1,00,<NA>
172678,F15Q10074428,2,00,<NA>
172679,F15Q10074428,3,00,<NA>
172680,F15Q10074428,4,00,<NA>
...,...,...,...,...
3396313,F15Q40302919,18,RA,<NA>
3396314,F15Q40302919,19,RA,<NA>
3396315,F15Q40302919,20,RA,<NA>
3396316,F15Q40302919,21,RA,<NA>


In [45]:
adverse_summary = (
    perf_24m.loc[perf_24m["loan_id"].isin(adverse_zbc_ids)]
    .assign(
        delq_numeric=lambda x: pd.to_numeric(
            x["current_loan_delinquency_status"],
            errors="coerce",
        )
    )
    .groupby("loan_id")
    .agg(
        max_delq=("delq_numeric", "max"),
        ever_90dpd=("delq_numeric", lambda x: x.ge(3).any()),
        last_age=("loan_age", "max"),
        zero_balance_code=("zero_balance_code", "last"),
    )
    .reset_index()
)

adverse_summary

,loan_id,max_delq,ever_90dpd,last_age,zero_balance_code
0,F15Q10074428,7,True,19,02
1,F15Q10168579,8,True,16,02
2,F15Q10202521,8,True,21,03
3,F15Q20079124,7,True,21,09
4,F15Q30019368,2,False,23,09
5,F15Q30134610,14,True,20,09
6,F15Q30194371,7,True,10,03
7,F15Q30254116,9,True,21,03
8,F15Q30257025,5,True,20,03
9,F15Q30276786,10,True,23,02


In [46]:
adverse_summary.groupby(["zero_balance_code", "ever_90dpd"]).size()

zero_balance_code  ever_90dpd
02                 True          3
03                 True          4
09                 False         1
                   True          5
dtype: int64

In [47]:
tmp = perf_24m.loc[perf_24m["loan_id"].isin(adverse_zbc_ids)].copy()

tmp["delq_numeric"] = pd.to_numeric(
    tmp["current_loan_delinquency_status"],
    errors="coerce",
)

tmp["is_serious_delq"] = tmp["delq_numeric"].ge(3) | tmp[
    "current_loan_delinquency_status"
].eq("RA")

adverse_summary = (
    tmp.groupby("loan_id")
    .agg(
        max_numeric_delq=("delq_numeric", "max"),
        ever_serious_delq=("is_serious_delq", "max"),
        last_age=("loan_age", "max"),
        zero_balance_code=("zero_balance_code", "last"),
    )
    .reset_index()
)

adverse_summary

,loan_id,max_numeric_delq,ever_serious_delq,last_age,zero_balance_code
0,F15Q10074428,7,True,19,02
1,F15Q10168579,8,True,16,02
2,F15Q10202521,8,True,21,03
3,F15Q20079124,7,True,21,09
4,F15Q30019368,2,True,23,09
5,F15Q30134610,14,True,20,09
6,F15Q30194371,7,True,10,03
7,F15Q30254116,9,True,21,03
8,F15Q30257025,5,True,20,03
9,F15Q30276786,10,True,23,02


In [48]:
tmp = perf_24m.loc[perf_24m["loan_id"].isin(adverse_zbc_ids)].copy()

tmp["delq_numeric"] = pd.to_numeric(
    tmp["current_loan_delinquency_status"],
    errors="coerce",
)

tmp["is_serious_delq"] = tmp["delq_numeric"].ge(3) | tmp[
    "current_loan_delinquency_status"
].eq("RA")

adverse_summary = (
    tmp.groupby("loan_id")
    .agg(
        max_numeric_delq=("delq_numeric", "max"),
        ever_serious_delq=("is_serious_delq", "max"),
        last_age=("loan_age", "max"),
        zero_balance_code=("zero_balance_code", "last"),
    )
    .reset_index()
)

adverse_summary

,loan_id,max_numeric_delq,ever_serious_delq,last_age,zero_balance_code
0,F15Q10074428,7,True,19,02
1,F15Q10168579,8,True,16,02
2,F15Q10202521,8,True,21,03
3,F15Q20079124,7,True,21,09
4,F15Q30019368,2,True,23,09
5,F15Q30134610,14,True,20,09
6,F15Q30194371,7,True,10,03
7,F15Q30254116,9,True,21,03
8,F15Q30257025,5,True,20,03
9,F15Q30276786,10,True,23,02


In [49]:
adverse_summary.groupby(["zero_balance_code", "ever_serious_delq"]).size()

zero_balance_code  ever_serious_delq
02                 True                 3
03                 True                 4
09                 True                 6
dtype: int64

In [50]:
special_zbc_ids = set(
    early_end_last.loc[
        early_end_last["zero_balance_code"].isin(["15", "96"])
        | early_end_last["zero_balance_code"].isna(),
        "loan_id",
    ]
)

special = perf_24m.loc[perf_24m["loan_id"].isin(special_zbc_ids)].copy()

special["delq_numeric"] = pd.to_numeric(
    special["current_loan_delinquency_status"],
    errors="coerce",
)

special["is_serious_delq"] = special["delq_numeric"].ge(3) | special[
    "current_loan_delinquency_status"
].eq("RA")

In [51]:
special_summary = (
    special.groupby("loan_id")
    .agg(
        first_age=("loan_age", "min"),
        last_age=("loan_age", "max"),
        max_numeric_delq=("delq_numeric", "max"),
        ever_serious_delq=("is_serious_delq", "max"),
        final_delq=("current_loan_delinquency_status", "last"),
        zero_balance_code=("zero_balance_code", "last"),
    )
    .reset_index()
)

special_summary

,loan_id,first_age,last_age,max_numeric_delq,ever_serious_delq,final_delq,zero_balance_code
0,F15Q10002812,0,11,0,False,00,96
1,F15Q10046347,0,22,18,True,18,15
2,F15Q10047986,1,8,0,False,00,96
3,F15Q10102794,0,7,0,False,00,96
4,F15Q10138012,0,7,0,False,00,96
5,F15Q10141205,0,8,0,False,00,96
6,F15Q10171273,0,4,0,False,00,96
7,F15Q10319507,1,9,0,False,00,96
8,F15Q10357702,12,20,0,False,00,96
9,F15Q10363877,0,12,4,True,04,<NA>


In [52]:
special_summary.groupby(
    ["zero_balance_code", "ever_serious_delq"],
    dropna=False,
).size()

zero_balance_code  ever_serious_delq
15                 True                  1
96                 False                27
                   True                  2
<NA>               True                  1
dtype: int64

In [54]:
first_observation = (
    perf.groupby("loan_id")
    .agg(
        first_loan_age=("loan_age", "min"),
        first_period=("period", "min"),
    )
    .reset_index()
)

first_observation["first_loan_age"].value_counts().sort_index()

first_loan_age
0     43842
1      3439
2       957
3       214
4        41
5        23
6        10
7         6
8        10
9         5
10        5
11        8
12        5
13        9
14       12
15        8
16        6
17        5
18        3
19        3
20        7
21        2
22        3
23       14
24       22
25       14
26        8
27        6
28        5
29        5
30        5
31        5
32        4
33        5
34        8
35       20
36       36
37       37
38       74
39       86
40       93
41       72
42       77
43      103
44       97
45       53
46       54
47       54
48       84
49       86
50       75
51       88
52       68
53       13
54        2
57        1
58        2
60        1
Name: count, dtype: Int64

In [55]:
first_observation.query("first_loan_age > 1")[
    "first_loan_age"
].value_counts().sort_index()

first_loan_age
2     957
3     214
4      41
5      23
6      10
7       6
8      10
9       5
10      5
11      8
12      5
13      9
14     12
15      8
16      6
17      5
18      3
19      3
20      7
21      2
22      3
23     14
24     22
25     14
26      8
27      6
28      5
29      5
30      5
31      5
32      4
33      5
34      8
35     20
36     36
37     37
38     74
39     86
40     93
41     72
42     77
43    103
44     97
45     53
46     54
47     54
48     84
49     86
50     75
51     88
52     68
53     13
54      2
57      1
58      2
60      1
Name: count, dtype: Int64

In [56]:
print("Start age 0:", (first_observation["first_loan_age"] == 0).sum())

print("Start age 1:", (first_observation["first_loan_age"] == 1).sum())

print("Start age 2-24:", first_observation["first_loan_age"].between(2, 24).sum())

print("Start age 25+:", (first_observation["first_loan_age"] >= 25).sum())

Start age 0: 43842
Start age 1: 3439
Start age 2-24: 1378
Start age 25+: 1341


In [57]:
(
    first_observation.loc[
        first_observation["first_loan_age"].between(2, 24),
        "first_loan_age",
    ]
    .value_counts()
    .sort_index()
)

first_loan_age
2     957
3     214
4      41
5      23
6      10
7       6
8      10
9       5
10      5
11      8
12      5
13      9
14     12
15      8
16      6
17      5
18      3
19      3
20      7
21      2
22      3
23     14
24     22
Name: count, dtype: Int64

In [58]:
start_vs_target = first_observation.merge(
    loan_target[["loan_id", "ever_90dpd_24m"]],
    on="loan_id",
    how="left",
)

start_vs_target["start_group"] = pd.cut(
    start_vs_target["first_loan_age"],
    bins=[-1, 0, 1, 5, 12, 24, float("inf")],
    labels=[
        "0",
        "1",
        "2-5",
        "6-12",
        "13-24",
        "25+",
    ],
)

(
    start_vs_target.groupby("start_group", observed=True)["ever_90dpd_24m"].agg(
        ["count", "sum", "mean"]
    )
)

,count,sum,mean
start_group,,,
0,43842,319,0.007276
1,3439,24,0.006979
2-5,1235,3,0.002429
6-12,49,0,0.0
13-24,94,0,0.0
25+,0,0,<NA>
